# MS1 ↔ Biomapper — exploration

Self-contained view over the latest run. Uses the same `report.aggregate()` the markdown
report uses, so numbers never drift, plus interactive drill-downs.
Run `python run_comparison.py` first. **Clear outputs before committing.**


In [ ]:
import glob, os, re
import pandas as pd
import compare as C, report as R, io_and_normalize as io

run = sorted(glob.glob('outputs/2026*'))[-1]
comp = pd.read_csv(os.path.join(run, 'comparison.csv'))
m = R.aggregate(comp)
print('run:', run)
print('features:', m['total'], '| name-only resolved:', m['lift']['name_only_resolved'])
print('hinted pass ran:', m['hinted_ran'])


## Concordance by namespace (name-only, authoritative)
Agreement over the *sized comparable denominator* (both sides have an ID).


In [ ]:
rows = []
for ns in R.ALL_NAMESPACES:
    d = m['namespaces'][ns]
    rows.append({'namespace': ns, 'comparable': d['comparable'],
                 'comparable_%': round(100*d['comparable_frac'], 1),
                 'exact_%': None if d['exact_rate'] is None else round(100*d['exact_rate'], 1),
                 'agreement_%': None if d['agreement_rate'] is None else round(100*d['agreement_rate'], 1),
                 'new_coverage': d['new_coverage'], 'missed': d['missed']})
pd.DataFrame(rows).set_index('namespace')


## Full concordance by tier (MS1 / MS2 / CURATION)

Same columns as the headline, computed within each tier.


In [ ]:
for t in m['tiers']:
    rows = []
    for ns in R.ALL_NAMESPACES:
        d = m['by_tier'][ns][t]
        rows.append({'namespace': ns, 'comparable': d['comparable'],
                     'exact_%': None if d['exact_rate'] is None else round(100*d['exact_rate'], 1),
                     'agreement_%': None if d['agreement_rate'] is None else round(100*d['agreement_rate'], 1),
                     'new_coverage': d['new_coverage'], 'missed': d['missed']})
    print(f"=== Tier: {t} ===")
    display(pd.DataFrame(rows).set_index('namespace'))


## Hinted-pass cross-namespace agreement (input-side hints)
Hints come from the **input** side only (HMDB parsed from MS1/MS2 names + CAS from `ms2_cas_id`),
never from the curated reference. Each namespace is scored only where it was *not* itself the hint
(circular cases excluded). Negative Δ means the (spectral) hint pulled Biomapper away from the curation.


In [ ]:
if not m['hinted_ran']:
    print('Hinted pass not run for this report.')
else:
    rows = []
    for ns in io.SCORED_NAMESPACES:
        d = m['hinted'].get(ns, {})
        no_a, hi_a = d.get('name_only_agreement'), d.get('hinted_agreement')
        rows.append({'namespace': ns,
                     'name_only_%': None if no_a is None else round(100*no_a, 1),
                     'hinted_%': None if hi_a is None else round(100*hi_a, 1),
                     'delta_pts': None if (no_a is None or hi_a is None) else round(100*(hi_a-no_a), 1),
                     'comparable': d.get('comparable', 0),
                     'excluded_circular': d.get('n_excluded_circular', 0)})
    display(pd.DataFrame(rows).set_index('namespace'))


## New coverage — UNVALIDATED candidates, by confidence tier
No ground truth where the reference has no ID. Spot-check before trusting.


In [ ]:
rows = []
for ns in R.ALL_NAMESPACES:
    by = m['new_coverage_by_conf'].get(ns) or {}
    rows.append({'namespace': ns, **{f'{k}_conf': v for k, v in sorted(by.items())}})
pd.DataFrame(rows).set_index('namespace').fillna(0)


## Class distribution per namespace


In [ ]:
pd.DataFrame({ns: comp[f'{ns}__class'].value_counts() for ns in io.SCORED_NAMESPACES}).fillna(0).astype(int)


## RefMet detail (Biomapper RefMet ID → name via MW REST, vs reference `refmet_name`)


In [ ]:
print('refmet class distribution:', comp['refmet__class'].value_counts().to_dict())
ref_dis = comp[comp['refmet__class'] == C.DISAGREE]
ref_dis[['feature_id','matched_name','refmet__ref','refmet__bmap_ids','confidence_tier']].head(20)


## Disagreements — drill down (change `ns`)


In [ ]:
ns = 'CHEBI'
dis = comp[comp[f'{ns}__class'] == C.DISAGREE]
dis[['feature_id','matched_name','match_level',f'{ns}__ref',f'{ns}__bmap','confidence_tier']]


## New-coverage browser (change `ns`)


In [ ]:
ns = 'PUBCHEM.COMPOUND'
nc = comp[comp[f'{ns}__class'] == C.NEW_COVERAGE]
print(nc['confidence_tier'].value_counts().to_dict())
nc[['feature_id','matched_name','match_level',f'{ns}__bmap','confidence_tier']].head(30)


## Per-method breakdown (from the xlsx)
A feature_id may appear in multiple method sheets.


In [ ]:
xls = pd.ExcelFile('data/All_Methods_Features.xlsx')
method_of = {}
for sheet in xls.sheet_names:
    s = xls.parse(sheet, usecols=['feature_id'])
    for fid in s['feature_id'].astype(str):
        method_of.setdefault(fid, set()).add(sheet)
mrows = []
for _, r in comp.iterrows():
    for method in method_of.get(str(r['feature_id']), {'(none)'}):
        mrows.append({'method': method, 'cls': r['CHEBI__class']})
mdf = pd.DataFrame(mrows)
agree = mdf['cls'].isin([C.AGREE_EXACT, C.AGREE_PARTIAL])
comparable = mdf['cls'].isin([C.AGREE_EXACT, C.AGREE_PARTIAL, C.DISAGREE])
summary = mdf.assign(agree=agree, comparable=comparable).groupby('method')[['agree','comparable']].sum()
summary['CHEBI_agreement_%'] = (100*summary['agree']/summary['comparable']).round(1)
summary
